<a href="https://colab.research.google.com/github/hdgaribay/Audio_Enhancement_CNN/blob/main/RF_Pipeline_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ RF → IQ Denoising → FM Demod → MUSE Pipeline

**Run these cells in order, top to bottom.**

## What this notebook does:
1. Mounts your Google Drive (so checkpoints survive disconnects)
2. Installs all dependencies
3. Downloads LibriSpeech dataset (free, clean speech audio)
4. Clones the MUSE repo
5. Uploads your pipeline code
6. Trains Stage 1 (IQ denoiser) — saves to Drive every epoch
7. Tests the full pipeline on a sample

⏱️ **Expected time on T4:** ~4–5 hours for 50 epochs

💾 **Checkpoints save to Google Drive** — if Colab disconnects,
just reopen and jump to the 'Resume Training' cell.

---
## Step 1 — Mount Google Drive
This saves your checkpoints to Drive so they survive Colab disconnects.
A popup will ask you to sign in — that's normal.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All checkpoints and outputs will be saved here on your Drive
DRIVE_DIR = '/content/drive/MyDrive/rf_speech_pipeline'
CKPT_DIR  = os.path.join(DRIVE_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'✅ Drive mounted. Checkpoints will save to:')
print(f'   {CKPT_DIR}')

Mounted at /content/drive
✅ Drive mounted. Checkpoints will save to:
   /content/drive/MyDrive/rf_speech_pipeline/checkpoints


---
## Step 2 — Check GPU
Make sure you have a T4 (or better). If this shows 'CPU only', go to:
Runtime → Change runtime type → GPU → T4

In [2]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}  |  VRAM: {vram:.1f} GB')
else:
    print('❌ No GPU found! Go to Runtime → Change runtime type → GPU')

✅ GPU: Tesla T4  |  VRAM: 15.6 GB


---
## Step 3 — Install Dependencies

In [3]:
# Install all required packages
# This takes about 2-3 minutes
!pip install -q soundfile noisereduce librosa torchaudio

print('✅ Dependencies installed')

✅ Dependencies installed


---
## Step 4 — Upload Your Pipeline Code
Upload these files from your computer:
- `rf_simulator.py`
- `stage1_iq_denoiser.py`
- `stage2_fm_demodulator.py`
- `stage3_muse_enhancement.py`
- `train_iq_denoiser.py`
- `pipeline.py`

In [4]:
from google.colab import files

print('Select all 6 .py files at once (Ctrl+click or Cmd+click):')
uploaded = files.upload()

# Move everything to /content/pipeline/
os.makedirs('/content/pipeline', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'/content/pipeline/{fname}')
    print(f'  ✅ {fname}')

# Add to Python path so files can import each other
import sys
sys.path.insert(0, '/content/pipeline')
print('\n✅ All files ready')

Select all 6 .py files at once (Ctrl+click or Cmd+click):


Saving pipeline.py to pipeline.py
Saving rf_simulator.py to rf_simulator.py
Saving stage1_iq_denoiser.py to stage1_iq_denoiser.py
Saving stage2_fm_demodulator.py to stage2_fm_demodulator.py
Saving stage3_muse_enhancement.py to stage3_muse_enhancement.py
Saving train_iq_denoiser.py to train_iq_denoiser.py
  ✅ pipeline.py
  ✅ rf_simulator.py
  ✅ stage1_iq_denoiser.py
  ✅ stage2_fm_demodulator.py
  ✅ stage3_muse_enhancement.py
  ✅ train_iq_denoiser.py

✅ All files ready


---
## Step 5 — Download LibriSpeech Dataset

LibriSpeech is a free dataset of clean English speech.
We use `train-clean-100` — 100 hours of audio, ~6 GB.

**What is a dataset?** Think of it as thousands of audio
examples the model will learn from. The more variety,
the better the model generalizes to new voices.

⏱️ Download takes about 10 minutes on Colab's connection.

In [12]:
import os

DATA_DIR = '/content/librispeech'
os.makedirs(DATA_DIR, exist_ok=True)

# Download and extract (train-clean-100 = 6.3 GB)
print('Downloading LibriSpeech train-clean-100 (~6 GB)...')
!wget -q --show-progress -P {DATA_DIR} \
    https://www.openslr.org/resources/12/train-clean-100.tar.gz

print('\nExtracting...')
!tar -xzf {DATA_DIR}/train-clean-100.tar.gz -C {DATA_DIR}
!rm {DATA_DIR}/train-clean-100.tar.gz   # free up disk space

# Count the audio files
import glob
flac_files = glob.glob(f'{DATA_DIR}/LibriSpeech/train-clean-100/**/*.flac', recursive=True)
print(f'\n✅ Found {len(flac_files):,} audio files ready for training')

train-clean-100.tar 100%[===================>]   5.95G  7.94MB/s    in 13m 0s  

Extracting...

✅ Found 28,539 audio files ready for training


---
## Step 6 — Clone MUSE Repo (for Stage 3)
This gets the pretrained MUSE model so Stage 3 is ready to go.

In [13]:
import os
os.chdir('/content/pipeline')

# Clone MUSE — but do NOT install its requirements.txt
# It pins torch==1.8.1+cu111 which no longer exists.
# Colab's existing PyTorch is modern and works fine.
!git clone https://github.com/huaidanquede/MUSE-Speech-Enhancement

# Only install the non-torch dependencies from MUSE
!pip install -q pesq pystoi librosa

# Verify the pretrained checkpoint is present
ckpt = 'MUSE-Speech-Enhancement/paper_result/g_best'
if os.path.isfile(ckpt):
    print(f'✅ MUSE checkpoint found')
else:
    print('⚠️  Checkpoint missing — Stage 3 will use lightweight fallback')

Cloning into 'MUSE-Speech-Enhancement'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 71 (delta 19), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 4.43 MiB | 17.64 MiB/s, done.
Resolving deltas: 100% (19/19), done.
  Preparing metadata (setup.py) ... done
✅ MUSE checkpoint found


---
## Step 7 — Train Stage 1 (IQ Denoiser)

This is the main training run. It will:
- Generate noisy IQ training pairs on-the-fly from LibriSpeech
- Train for 50 epochs (~4–5 hours on T4)
- Save a checkpoint after every epoch to Google Drive
- Print train/val loss each epoch so you can watch progress

**What is loss?** A number measuring how wrong the model is.
You want to see it go DOWN over epochs. It won't go to zero —
around 0.05–0.15 is good for this task.

⚠️ **Don't close this tab!** The cell runs for hours.
If Colab disconnects, use the Resume cell below.

In [ ]:
os.chdir('/content/pipeline')

!python train_iq_denoiser.py \
    --audio_dir /content/librispeech/LibriSpeech/train-clean-100/ \
    --checkpoint_dir {CKPT_DIR} \
    --epochs 50 \
    --batch_size 8 \
    --segment_sec 0.1 \
    --lr 2e-4

Streaming output truncated to the last 5000 lines.
              FM deviation   : ±25 kHz
  [simulator] generated FM IQ signal
              audio duration : 0.10s
              IQ samples     : 19,200
              RF sample rate : 200,000 Hz
              SNR            : 16.248979874280202 dB
              FM deviation   : ±25 kHz
  [simulator] generated FM IQ signal
              audio duration : 0.10s
              IQ samples     : 19,200
              RF sample rate : 200,000 Hz
              SNR            : 99.0 dB
              FM deviation   : ±25 kHz
  [simulator] generated FM IQ signal
              audio duration : 0.10s
              IQ samples     : 19,200
              RF sample rate : 200,000 Hz
              SNR            : 7.946654196800567 dB
              FM deviation   : ±25 kHz
  [simulator] generated FM IQ signal
              audio duration : 0.10s
              IQ samples     : 19,200
              RF sample rate : 200,000 Hz
              SNR            : 99

---
## Step 7b — Resume Training (after a disconnect)

If Colab disconnected, run **Steps 1–6** again to remount Drive
and reinstall packages, then run this cell instead of Step 7.
It picks up from the last saved epoch automatically.

In [14]:
# Run this as a new cell in Colab — it patches rf_simulator.py in place
import re

with open('/content/pipeline/rf_simulator.py', 'r') as f:
    code = f.read()

# Remove the print block from generate_fm_iq_signal
old = """    print(f"  [simulator] generated FM IQ signal")
    print(f"              audio duration : {len(audio)/audio_sr:.2f}s")
    print(f"              IQ samples     : {len(iq_noisy):,}")
    print(f"              RF sample rate : {RF_SAMPLE_RATE:,} Hz")
    print(f"              SNR            : {snr_db} dB")
    print(f"              FM deviation   : ±{FM_DEVIATION/1000:.0f} kHz")

    return iq_noisy, RF_SAMPLE_RATE"""

new = """    return iq_noisy, RF_SAMPLE_RATE"""

code = code.replace(old, new)

with open('/content/pipeline/rf_simulator.py', 'w') as f:
    f.write(code)

print("✅ rf_simulator.py patched — prints removed")

✅ rf_simulator.py patched — prints removed


In [8]:
# Install missing system library for reading .flac files
!apt-get install -q libsndfile1
!pip install -q soundfile

print("✅ libsndfile installed")

Reading package lists...
Building dependency tree...
Reading state information...
libsndfile1 is already the newest version (1.0.31-2ubuntu0.2).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
✅ libsndfile installed


In [10]:
# Patch the dataset to use librosa instead of soundfile for loading audio
with open('/content/pipeline/train_iq_denoiser.py', 'r') as f:
    code = f.read()

old = """        audio, sr = sf.read(self.audio_files[file_idx], dtype="float32")
        if audio.ndim == 2: audio = audio.mean(axis=1)
        audio = audio / (np.max(np.abs(audio)) + 1e-8)"""

new = """        import librosa
        audio, sr = librosa.load(str(self.audio_files[file_idx]), sr=16000, mono=True)
        audio = audio / (np.max(np.abs(audio)) + 1e-8)"""

code = code.replace(old, new)
with open('/content/pipeline/train_iq_denoiser.py', 'w') as f:
    f.write(code)

print("✅ Patched — now using librosa to load audio")

✅ Patched — now using librosa to load audio


In [ ]:
os.chdir('/content/pipeline')

LATEST_CKPT = os.path.join(CKPT_DIR, 'iq_denoiser_latest.pt')

if os.path.isfile(LATEST_CKPT):
    # Check what epoch we're resuming from
    import torch
    info = torch.load(LATEST_CKPT, map_location='cpu')
    last_epoch = info.get('epoch', '?')
    best_val   = info.get('best_val', '?')
    print(f'▶ Resuming from epoch {last_epoch}  |  best val loss so far: {best_val}')

    !python train_iq_denoiser.py \
        --audio_dir /content/librispeech/LibriSpeech/train-clean-100/ \
        --checkpoint_dir {CKPT_DIR} \
        --resume {LATEST_CKPT} \
        --epochs 50 \
        --batch_size 8 \
        --segment_sec 0.1 \
        --lr 2e-4
else:
    print('❌ No checkpoint found — run Step 7 first to start training from scratch')

▶ Resuming from epoch 1  |  best val loss so far: inf
  Training IQ denoiser on: cuda

  Train files: 25685  |  Val files: 2854

  [dataset] 25685 audio files  |  segment: 0.1s
  [dataset] 2854 audio files  |  segment: 0.1s
  Model: 4.16M parameters

  ▶ Resumed from epoch 1  |  best_val=inf
  Epoch 2 | Step 100/32107 | Loss 0.0198
  Epoch 2 | Step 200/32107 | Loss 0.0259
  Epoch 2 | Step 300/32107 | Loss 0.0299
  Epoch 2 | Step 400/32107 | Loss 0.0246
  Epoch 2 | Step 500/32107 | Loss 0.0240
  Epoch 2 | Step 600/32107 | Loss 0.0279
  Epoch 2 | Step 700/32107 | Loss 0.0246
  Epoch 2 | Step 800/32107 | Loss 0.0180
  Epoch 2 | Step 900/32107 | Loss 0.0306
  Epoch 2 | Step 1000/32107 | Loss 0.0249
  Epoch 2 | Step 1100/32107 | Loss 0.0197
  Epoch 2 | Step 1200/32107 | Loss 0.0288
  Epoch 2 | Step 1300/32107 | Loss 0.0348
  Epoch 2 | Step 1400/32107 | Loss 0.0251
  Epoch 2 | Step 1500/32107 | Loss 0.0297
  Epoch 2 | Step 1600/32107 | Loss 0.0180
  Epoch 2 | Step 1700/32107 | Loss 0.0355
  

---
## Step 8 — Test the Full Pipeline

Once training is done (or even mid-training to check quality),
run the full 3-stage pipeline on a sample audio clip.

We'll grab a LibriSpeech sample as our test speech,
simulate an RF transmission, run all 3 stages, and
listen to the result.

In [ ]:
import glob, random

# Pick a random LibriSpeech file as test audio
test_files = glob.glob('/content/librispeech/LibriSpeech/train-clean-100/**/*.flac',
                       recursive=True)
test_audio = random.choice(test_files)
print(f'Test audio: {test_audio}')

# Convert .flac to .wav (pipeline expects .wav)
!ffmpeg -i "{test_audio}" /content/pipeline/test_input.wav -y -loglevel quiet
print('✅ Test file ready: /content/pipeline/test_input.wav')

In [ ]:
os.chdir('/content/pipeline')

BEST_CKPT   = os.path.join(CKPT_DIR, 'iq_denoiser_best.pt')
MUSE_CKPT   = 'MUSE-Speech-Enhancement/paper_result/g_best'

!python pipeline.py \
    --audio test_input.wav \
    --snr 10 \
    --iq_checkpoint {BEST_CKPT} \
    --muse_checkpoint {MUSE_CKPT} \
    --output output/final_enhanced.wav \
    --save_stages

print('\n✅ Pipeline complete!')
print('Output files:')
!ls -lh output/


══════════════════════════════════════════════════════
  RF → IQ Denoising → FM Demod → MUSE Enhancement
══════════════════════════════════════════════════════

▶ Simulating FM transmission...
  [simulator] generated FM IQ signal
              audio duration : 15.58s
              IQ samples     : 2,991,360
              RF sample rate : 200,000 Hz
              SNR            : 10.0 dB
              FM deviation   : ±25 kHz
  [save_iq] saved 2,991,360 IQ samples → 'output/stage0_iq_noisy.iq'

▶ Stage 1 — IQ Signal Denoising (ML)
  (Cleaning noise from the raw RF signal before demodulation)
Traceback (most recent call last):
  File "/content/pipeline/pipeline.py", line 129, in <module>
    run_pipeline(args)
  File "/content/pipeline/pipeline.py", line 77, in run_pipeline
    iq_denoiser = load_iq_denoiser(args.iq_checkpoint)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/pipeline/stage1_iq_denoiser.py", line 250, in load_iq_denoiser
    model.load_state_dict(

---
## Step 9 — Listen to the Results
Compare audio at each stage to hear the improvement.

In [ ]:
import IPython.display as ipd
import soundfile as sf

def play(path, label):
    if os.path.isfile(path):
        audio, sr = sf.read(path)
        print(f'\n🔊 {label}')
        display(ipd.Audio(audio, rate=sr))
    else:
        print(f'  (not found: {path})')

play('test_input.wav',                '0. Original clean speech (before RF simulation)')
play('output/stage2_demodulated.wav', '1. After FM demodulation (with noise)')
play('output/final_enhanced.wav',     '2. Final output (after all 3 stages)')


🔊 0. Original clean speech (before RF simulation)


  (not found: output/stage2_demodulated.wav)
  (not found: output/final_enhanced.wav)


---
## Step 10 — Download Your Trained Model
Save the best checkpoint to your computer.

In [ ]:
from google.colab import files

BEST = os.path.join(CKPT_DIR, 'iq_denoiser_best.pt')
if os.path.isfile(BEST):
    print('Downloading best checkpoint...')
    files.download(BEST)
else:
    print('❌ No checkpoint found yet — complete at least 1 epoch of training first')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 📊 Monitor Training Progress
Run this cell any time to see a loss curve of training so far.

*(This cell reads the checkpoint file to extract the loss history)*

In [ ]:
import torch, glob
import matplotlib.pyplot as plt

latest = os.path.join(CKPT_DIR, 'iq_denoiser_latest.pt')
if os.path.isfile(latest):
    info = torch.load(latest, map_location='cpu')
    print(f"Last completed epoch : {info.get('epoch', '?')}")
    print(f"Best validation loss : {info.get('best_val', '?'):.4f}")
    print("\nTip: val loss < 0.15 = good  |  < 0.08 = very good")
else:
    print('No checkpoint yet — training has not started or Drive is not mounted')

Last completed epoch : 1
Best validation loss : inf

Tip: val loss < 0.15 = good  |  < 0.08 = very good
